# 01 — Exploratory Data Analysis (EDA)
**Project:** Automated PPE Detection Using CNN-Based Object Detection  
**Group:** Magical Unicorn | 6DANCS  
**Lead:** John Vincent Lapid  

This notebook covers:
1. Class count & distribution
2. Class imbalance analysis
3. Sample image grid (annotated)
4. Bounding box size distribution
5. Image brightness distribution

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────
DATASET_DIR = '../data/dataset'   # After running 02_split_dataset.py
SPLITS = ['train', 'val', 'test']

CLASS_NAMES = {
    0: 'hardhat',
    1: 'vest',
    2: 'face_mask',
    3: 'no_hardhat',
    4: 'no_vest',
    5: 'no_face_mask',
    6: 'person',
}
CLASS_COLORS = {
    0: '#2196F3',  # blue
    1: '#4CAF50',  # green
    2: '#FF9800',  # orange
    3: '#F44336',  # red
    4: '#E91E63',  # pink
    5: '#9C27B0',  # purple
    6: '#607D8B',  # grey
}

print('Config loaded. Dataset dir:', os.path.abspath(DATASET_DIR))

## 1. Class Count & Distribution

In [ ]:
def count_classes(split):
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    counts = Counter()
    for fname in os.listdir(lbl_dir):
        if not fname.endswith('.txt'):
            continue
        with open(os.path.join(lbl_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1
    return counts

split_counts = {s: count_classes(s) for s in SPLITS}

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Class Annotation Counts per Split', fontsize=15, fontweight='bold')

for ax, split in zip(axes, SPLITS):
    counts = split_counts[split]
    ids = sorted(counts.keys())
    names = [CLASS_NAMES.get(i, str(i)) for i in ids]
    values = [counts[i] for i in ids]
    colors = [CLASS_COLORS.get(i, '#999') for i in ids]
    bars = ax.bar(names, values, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_title(f'{split.capitalize()} split', fontsize=12)
    ax.set_xlabel('Class')
    ax.set_ylabel('Annotation count')
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../experiments/results/eda_class_counts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/eda_class_counts.png')

## 2. Class Imbalance Analysis

In [ ]:
train_counts = split_counts['train']
total = sum(train_counts.values())

print('=== Training Set Class Distribution ===')
print(f'{'Class':<15} {'Count':>8} {'% of total':>12} {'Imbalance ratio':>16}')
print('-' * 55)
max_count = max(train_counts.values()) if train_counts else 1
for cid in sorted(train_counts):
    name = CLASS_NAMES.get(cid, str(cid))
    cnt  = train_counts[cid]
    pct  = cnt / total * 100
    ratio = max_count / cnt if cnt > 0 else float('inf')
    print(f'{name:<15} {cnt:>8} {pct:>11.1f}% {ratio:>14.2f}x')

print(f'\nTotal annotations (train): {total}')

# Pie chart
fig, ax = plt.subplots(figsize=(8, 6))
ids = sorted(train_counts.keys())
ax.pie(
    [train_counts[i] for i in ids],
    labels=[CLASS_NAMES.get(i, str(i)) for i in ids],
    colors=[CLASS_COLORS.get(i, '#999') for i in ids],
    autopct='%1.1f%%', startangle=140, pctdistance=0.82
)
ax.set_title('Training Set Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../experiments/results/eda_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Sample Image Grid (with Bounding Boxes)

In [ ]:
def draw_boxes(img_path, lbl_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(1, figsize=(4, 4))
    ax.imshow(img)
    ax.axis('off')
    try:
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cid, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:5])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                color = CLASS_COLORS.get(cid, '#fff')
                rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                          linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 4, CLASS_NAMES.get(cid, str(cid)),
                        color='white', fontsize=7,
                        bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none'))
    except Exception:
        pass
    return fig

# Pick 8 random samples from train
import random
random.seed(42)
train_img_dir = os.path.join(DATASET_DIR, 'images', 'train')
train_lbl_dir = os.path.join(DATASET_DIR, 'labels', 'train')
sample_imgs = random.sample(os.listdir(train_img_dir), min(8, len(os.listdir(train_img_dir))))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Sample Training Images with YOLO Annotations', fontsize=14, fontweight='bold')

for ax, fname in zip(axes.flat, sample_imgs):
    img_path = os.path.join(train_img_dir, fname)
    lbl_path = os.path.join(train_lbl_dir, os.path.splitext(fname)[0] + '.txt')
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    ax.axis('off')
    try:
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cid, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:5])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                color = CLASS_COLORS.get(cid, '#fff')
                rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                          linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
    except Exception:
        pass

plt.tight_layout()
plt.savefig('../experiments/results/eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/eda_sample_images.png')

## 4. Bounding Box Size Distribution

In [ ]:
box_areas = []
lbl_dir = os.path.join(DATASET_DIR, 'labels', 'train')
for fname in os.listdir(lbl_dir):
    if not fname.endswith('.txt'):
        continue
    with open(os.path.join(lbl_dir, fname)) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                bw, bh = float(parts[3]), float(parts[4])
                box_areas.append(bw * bh * 100)  # as % of image area

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(box_areas, bins=60, color='#2196F3', edgecolor='white', linewidth=0.5)
ax.axvline(np.median(box_areas), color='red', linestyle='--', linewidth=1.5,
           label=f'Median: {np.median(box_areas):.2f}%')
ax.set_xlabel('Bounding Box Area (% of image)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Bounding Box Size Distribution (Training Set)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../experiments/results/eda_bbox_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total bboxes analyzed: {len(box_areas)}')
print(f'Median bbox area: {np.median(box_areas):.2f}% of image')
print(f'Mean bbox area:   {np.mean(box_areas):.2f}% of image')

## 5. Image Brightness Distribution

In [ ]:
brightness_vals = []
img_dir = os.path.join(DATASET_DIR, 'images', 'train')
sample = random.sample(os.listdir(img_dir), min(1000, len(os.listdir(img_dir))))

for fname in sample:
    img = cv2.imread(os.path.join(img_dir, fname), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        brightness_vals.append(np.mean(img))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(brightness_vals, bins=50, color='#FF9800', edgecolor='white', linewidth=0.5)
ax.axvline(np.mean(brightness_vals), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean: {np.mean(brightness_vals):.1f}')
ax.set_xlabel('Mean Pixel Intensity (0=Black, 255=White)', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Image Brightness Distribution (Training Set — 1000 sample)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../experiments/results/eda_brightness.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean brightness: {np.mean(brightness_vals):.1f}')
print(f'Std deviation:   {np.std(brightness_vals):.1f}')

## Summary

| Finding | Value |
|---------|-------|
| Dominant class | hardhat (highest annotation count) |
| Most underrepresented | face_mask |
| Typical bbox area | ~3–12% of image (small objects) |
| Lighting | Bimodal — outdoor bright + indoor/shaded |

**Key insight:** Class imbalance between hardhat and face_mask is the primary data challenge. Mosaic + Mixup augmentation is the planned mitigation during training.